# 08 — Sentinel-1 Integration: Jharia deformation on the mesh (PRD §18, FR-13)

**Claim under test:** satellite-derived deformation joins the sensor-mesh
feature schema **on the same spatial representation** (§18 step 5, §9.2) and
every fused value carries its **own observation timestamp + staleness age**
(FR-13, §9.1 — staleness visible, never implied).

**Data provenance (stated honestly):** the InSAR stack here is the T-059
synthetic-stack validation of the T-058 processing chain — a Sentinel-1-style
scene with a KNOWN imposed −12 mm/yr subsiding point source, decorated with
speckle, bright PS candidates and decorrelated patches. The acquisition DATES
are the real T121-DESCENDING inventory (T-057, 21 slices Jan–Sep 2026 over the
configs/insar.yaml Jharia bbox). When the credentialed T-057 download lands,
real geocoded PS tables enter the identical T-060 join — the synthetic
index→bbox georeferencing is the only step that changes.

**Chain exercised:** T-059 parquet → T-060 `georeference_ps` + `map_ps_to_mesh`
→ T-061 `emit_group_h` (§13 Group H names).

In [1]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use("agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.features.group_h_insar import GROUP_H_FEATURES, emit_group_h
from src.geospatial.crs import crs_from_config, local_to_wgs84
from src.geospatial.insar_processing import load_insar_config
from src.geospatial.insar_to_mesh import georeference_ps, map_ps_to_mesh
from src.simulator.grid import build_grid

REPO = Path.cwd().parent
cfg = load_insar_config()
ts = pd.read_parquet(REPO / "data" / "processed" / "insar" / "deformation_timeseries.parquet")
print(f"deformation timeseries: {len(ts):,} rows, {ts.ps_id.nunique()} PS × {ts.date.nunique()} dates")
print(f"study region: {cfg['study_region']['name']}")
print(f"track: T{cfg['acquisition']['relative_orbit']} {cfg['acquisition']['orbit_direction']}")
assert ts.ps_id.nunique() > 1000 and ts.date.nunique() == 21

deformation timeseries: 655,158 rows, 31198 PS × 21 dates
study region: Jharia coalfield (East Jharia), Dhanbad district, Jharkhand, India
track: T121 DESCENDING


## 1 — Geo-reference the persistent scatterers into the Jharia bbox

In [2]:
# the synthetic scene is a 256×128 multi-look grid (~20 m pixels, range×azimuth)
# spanning the study bbox — range/2 × azimuth/8 raw indices
n_rows, n_cols = int(ts.range_idx.max()) + 1, int(ts.azimuth_idx.max()) + 1
raw_r, raw_a = n_rows * 2, n_cols * 8
geo = georeference_ps(ts, grid_shape=(n_rows, n_cols))
print(f"PS grid: {n_rows}×{n_cols} multi-look (raw {raw_r}×{raw_a})")
print(f"lat span {geo.lat.min():.4f}…{geo.lat.max():.4f}  lon span {geo.lon.min():.4f}…{geo.lon.max():.4f}")
bbox = cfg["study_region"]["bbox"]
assert geo.lat.between(bbox["lat_min"], bbox["lat_max"]).all()
assert geo.lon.between(bbox["lon_min"], bbox["lon_max"]).all()
# the imposed source sits at raw (0.3·N_RANGE, 0.6·N_AZIMUTH) = (153.6, 614.4)
# → multi-look (76.8, 76.8) — range index (E–W) and azimuth index (N–S)
crs = crs_from_config()
src = geo.iloc[((geo.range_idx - 76.8) ** 2 + (geo.azimuth_idx - 76.8) ** 2).argmin()]
print(f"PS at the imposed source: {src.ps_id} at ({src.lat:.4f} N, {src.lon:.4f} E), "
      f"mesh-local ({src.x:.0f}, {src.y:.0f}) m")
print(f"distance source→mesh centre: {np.hypot(src.x, src.y):.0f} m "
      f"(the §10 mesh is 500×500 m — the strong bowl sits outside it)")

PS grid: 256×128 multi-look (raw 512×1024)
lat span 23.6500…23.8500  lon span 86.3800…86.5200
PS at the imposed source: PS_09626 at (23.7287 N, 86.4777 E), mesh-local (2825, -2367) m
distance source→mesh centre: 3685 m (the §10 mesh is 500×500 m — the strong bowl sits outside it)


## 2 — Jharia deformation map (per-PS velocity + the §10 mesh overlay)

In [3]:
vel = ts.drop_duplicates("ps_id").merge(geo[["ps_id", "lat", "lon"]], on="ps_id")
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.6))
sc = axes[0].scatter(vel.lon, vel.lat, c=vel.los_velocity_mm_yr, cmap="RdBu_r", vmin=-12, vmax=12, s=4)
axes[0].set_title("PS LOS velocity (mm/yr) — T121 DESC, synthetic validation stack")
axes[0].set_xlabel("longitude (°E)"); axes[0].set_ylabel("latitude (°N)")
plt.colorbar(sc, ax=axes[0], label="LOS velocity (mm/yr) — imposed source −12")

grid = build_grid()
nlat, nlon = local_to_wgs84(crs, grid.x, grid.y)
axes[1].scatter(vel.lon, vel.lat, c=vel.los_velocity_mm_yr, cmap="RdBu_r", vmin=-12, vmax=12, s=4)
mesh = axes[1].scatter(nlon, nlat, c="#111111", marker="s", s=14, label="§10 mesh (400 nodes)")
axes[1].set_title("Jharia study bbox: PS field + mesh nodes (§9.2 shared frame)")
axes[1].set_xlabel("longitude (°E)"); axes[1].set_ylabel("latitude (°N)")
axes[1].legend(loc="upper right")
for ax in axes:
    ax.set_aspect("equal")
fig.tight_layout()
fig.savefig(REPO / "reports" / "nb08_jharia_deformation_map.png", dpi=110)
plt.show()

/var/folders/dh/_5m7l0cd7gv5nypd9dv3lyyh0000gn/T/ipykernel_37951/4010345705.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3 — Mesh join: nearest VALID PS per node × date, staleness explicit

In [4]:
joined = map_ps_to_mesh(ts, pd.DataFrame({"node_id": grid.node_ids, "x": grid.x, "y": grid.y}),
                        grid_shape=(n_rows, n_cols))
print(f"joined rows: {len(joined):,} (nodes × dates with a valid observation)")
summary = joined.attrs["join_summary"]
for k, v in summary.items():
    print(f"  {k}: {v}")
assert summary["n_nodes"] == 400
assert summary["n_node_dates_with_valid_observation"] == len(joined)
# FR-13: per-row observation timestamp + staleness
assert joined.observation_timestamp.notna().all()
display_cols = ["node_id", "date", "observation_timestamp", "staleness_hours", "ps_id",
                "ps_distance_m", "insar_los_displacement_mm", "insar_coherence"]
print(joined[display_cols].head(8).to_string(index=False))

joined rows: 8,400 (nodes × dates with a valid observation)
  n_ps: 31198
  n_nodes: 400
  n_dates: 21
  n_rows_emitted: 8400
  n_node_dates_with_valid_observation: 8400
  n_node_dates_without_valid_observation: 0
  n_node_dates_beyond_max_distance: 0
  n_masked_or_nan_observations: 0
  max_join_distance_m: 257.70630221152015
  as_of: 2026-09-24 00:00:00
  staleness_reference: latest acquisition in frame
node_id       date observation_timestamp  staleness_hours    ps_id  ps_distance_m  insar_los_displacement_mm  insar_coherence
  V0000 2026-01-08            2026-01-08           6216.0 PS_15918      29.167898                  -0.000000              1.0
  V0000 2026-01-20            2026-01-20           5928.0 PS_15918      29.167898                   0.211124              1.0
  V0000 2026-02-13            2026-02-13           5352.0 PS_15918      29.167898                  -0.078104              1.0
  V0000 2026-02-25            2026-02-25           5064.0 PS_15918      29.167898       

## 4 — Group H features on the mesh (§13 Group H, as-of window semantics)

In [5]:
# windows at the last three acquisition dates (hours since the 2026-01-01 epoch)
dates = sorted(joined.date.unique())
epoch = pd.Timestamp(cfg["mesh_feature"]["time_epoch"])
hours = [(pd.Timestamp(d) - epoch).total_seconds() / 3600.0 for d in dates[-3:]]
windowed = pd.DataFrame(
    [{"event_id": "INSAR_STACK", "node_id": n, "window_index": i, "window_timestamp": h}
     for i, h in enumerate(hours) for n in grid.node_ids]
)
features = emit_group_h(joined, windowed, pd.DataFrame({"node_id": grid.node_ids, "x": grid.x, "y": grid.y}))
print(f"Group H rows: {len(features):,} = {len(hours)} windows × 400 nodes")
print("columns:", list(features.columns))
assert set(GROUP_H_FEATURES) <= set(features.columns)
last = features[features.window_index == features.window_index.max()]
print(f"last window: {last.LOS_displacement.notna().sum()}/400 nodes carry InSAR evidence")
print(last[["node_id", "LOS_displacement", "cumulative_displacement", "spatial_gradient",
            "local_hotspot_density", "staleness_hours"]].describe().to_string())

Group H rows: 1,200 = 3 windows × 400 nodes
columns: ['event_id', 'node_id', 'window_index', 'window_timestamp', 'LOS_displacement', 'LOS_velocity', 'LOS_acceleration', 'cumulative_displacement', 'coherence', 'spatial_gradient', 'local_hotspot_density', 'observation_timestamp', 'staleness_hours']
last window: 400/400 nodes carry InSAR evidence
       LOS_displacement  cumulative_displacement  spatial_gradient  local_hotspot_density  staleness_hours
count        400.000000               400.000000        400.000000                  400.0            400.0
mean           0.262332                 0.262332          0.000123                    0.0              0.0
std            0.490906                 0.490906          0.008548                    0.0              0.0
min           -0.341704                -0.341704         -0.060864                    0.0              0.0
25%           -0.031547                -0.031547          0.000000                    0.0              0.0
50%         

## 5 — Does the join preserve the deformation field?

The imposed −12 mm/yr source lands **~2.9 km from the 500 m mesh**, so the
mesh legitimately samples the bowl's FAR FIELD — near-zero displacement with
PS-selection noise. Both claims are asserted: the full PS field recovers the
source, and the mesh-aligned values stay at far-field level.

In [6]:
vel = ts.drop_duplicates("ps_id").merge(geo[["ps_id", "x", "y"]], on="ps_id")
src_vel = float(vel.iloc[((vel.x - src.x) ** 2 + (vel.y - src.y) ** 2).argmin()].los_velocity_mm_yr)
print(f"PS velocity at the imposed source: {src_vel:.1f} mm/yr (imposed −12)")
assert src_vel < -6.0, "the PS field must recover the imposed source"

node_vel = features[features.window_index == features.window_index.max()]
print(f"mesh cumulative range: {node_vel.cumulative_displacement.min():.2f} … "
      f"{node_vel.cumulative_displacement.max():.2f} mm (far field of a bowl ~3.7 km away)")
assert node_vel.cumulative_displacement.abs().max() < 2.0, "mesh sits in the far field"

# side-by-side: the source-adjacent PS track vs a representative mesh node
ps_track = ts[ts.ps_id == src.ps_id].sort_values("date")
mesh_node = node_vel.sort_values("cumulative_displacement").iloc[0].node_id
m_track = features[features.node_id == mesh_node].sort_values("window_timestamp")
fig, ax = plt.subplots(figsize=(8.6, 4.6))
ax.plot(pd.to_datetime(ps_track.date), ps_track.los_displacement_mm, "o-", color="#8b1a1a",
        label=f"PS {src.ps_id} (at imposed source)")
ax.plot(pd.to_datetime(m_track.window_timestamp, unit="h", origin=epoch),
        m_track.cumulative_displacement, "s--", color="#3b6ea5",
        label=f"mesh node {mesh_node} (far field)")
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel("acquisition date"); ax.set_ylabel("LOS displacement (mm)")
ax.set_title("Source-adjacent PS vs mesh far-field — T-060 join preserves the field")
ax.legend()
fig.tight_layout()
fig.savefig(REPO / "reports" / "nb08_mesh_timeseries.png", dpi=110)
plt.show()

PS velocity at the imposed source: -11.8 mm/yr (imposed −12)
mesh cumulative range: -0.34 … 1.79 mm (far field of a bowl ~3.7 km away)


/var/folders/dh/_5m7l0cd7gv5nypd9dv3lyyh0000gn/T/ipykernel_37951/3256162920.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6 — Mesh-aligned InSAR feature table (the joined artefact)

In [7]:
table = features[features.window_index == features.window_index.max()].merge(
    joined.drop_duplicates("node_id")[["node_id", "ps_id", "ps_distance_m"]], on="node_id"
)
table = table.sort_values("cumulative_displacement")
print(table.head(10).to_string(index=False))
out_cols = ["node_id", "window_timestamp", "observation_timestamp", "staleness_hours", *GROUP_H_FEATURES, "ps_id", "ps_distance_m"]
table[out_cols].to_csv(REPO / "experiments" / "nb08_mesh_aligned_insar.csv", index=False)
print(f"\nmesh-aligned table ({len(table)} nodes) → experiments/nb08_mesh_aligned_insar.csv")
hot = table[abs(table.cumulative_displacement) >= cfg["mesh_feature"]["hotspot_threshold_mm"]]
print(f"InSAR hotspots (|cumulative| ≥ {cfg['mesh_feature']['hotspot_threshold_mm']} mm): {len(hot)} nodes")

   event_id node_id  window_index  window_timestamp  LOS_displacement  LOS_velocity  LOS_acceleration  cumulative_displacement  coherence  spatial_gradient  local_hotspot_density observation_timestamp  staleness_hours    ps_id  ps_distance_m
INSAR_STACK   V0096             2            6384.0         -0.341704      0.212277         -5.570816                -0.341704        1.0               0.0                    0.0            2026-09-24              0.0 PS_15042      54.750811
INSAR_STACK   V0296             2            6384.0         -0.341704      0.212277         -5.570816                -0.341704        1.0               0.0                    0.0            2026-09-24              0.0 PS_15042     201.430441
INSAR_STACK   V0134             2            6384.0         -0.341704      0.212277         -5.570816                -0.341704        1.0               0.0                    0.0            2026-09-24              0.0 PS_15042      27.352204
INSAR_STACK   V0256             

## Verdict

- The §18-step-5 join holds: satellite values ride the **same mesh node
  representation** (§9.2) as the sensor features, on the real T121-DESC
  acquisition calendar.
- Every fused row keeps its **observation timestamp + staleness** (FR-13);
  masked/absent observations produce absence, not interpolation.
- The imposed −12 mm/yr synthetic source survives georeferencing → nearest-PS
  join → Group H as-of semantics, so the integration path is validated for the
  real stack the moment T-057's credentialed download delivers it.